In [1]:
import pandas as pd

Labour_hour = pd.read_excel(r"C:\Users\Admin\OneDrive - neousys-tech\Desktop\Laborhours.xlsx", sheet_name='Reference')

Labour_hour["pn"] = Labour_hour["pn"].astype(str).str.strip().str.upper()

labour_map = Labour_hour.set_index("pn").to_dict("index")

labour_map

# # all part numbers in labour_map
# for pn in labour_map.keys():
#     print(pn)


{'FLYC-300-EC-JON16-NS': {'sop': nan,
  'unitsinabox': 25,
  'buildpoints': 6,
  'testonlypoints': 2,
  'customized': 0,
  'gpu': 0,
  'extra': 0.0,
  'Part Number': nan,
  'Unit in workorder': nan,
  'Assemble': nan,
  'gpu.1': nan,
  'sop.1': nan},
 'N10208GC-10G-ORI02': {'sop': 'Nuvo-10208GC-Outrider-Installation.pptx',
  'unitsinabox': 1,
  'buildpoints': 15,
  'testonlypoints': 7,
  'customized': 2,
  'gpu': 3,
  'extra': 0.0,
  'Part Number': nan,
  'Unit in workorder': nan,
  'Assemble': nan,
  'gpu.1': nan,
  'sop.1': nan},
 'N8108GC-QD-NM0X': {'sop': 'NoahMedical-8108GC-QD-illustration-SOP.pptx',
  'unitsinabox': 1,
  'buildpoints': 16,
  'testonlypoints': 5,
  'customized': 2,
  'gpu': 3,
  'extra': 5.0,
  'Part Number': nan,
  'Unit in workorder': nan,
  'Assemble': nan,
  'gpu.1': nan,
  'sop.1': nan},
 'N9160GC-POE-I7DC-4060-CYN02-25': {'sop': 'SF011-A-N9160GC-PoE-i7DC-4060-CYN02-25.V1.1.041224.pdf',
  'unitsinabox': 3,
  'buildpoints': 9,
  'testonlypoints': 5,
  'customi

In [3]:
keep_cols = ["pn", "unitsinabox", "buildpoints", "testonlypoints", "gpu", "sop", "extra"]
clean_df = Labour_hour[keep_cols].copy()
clean_df["pn"] = clean_df["pn"].astype(str).str.strip().str.upper()

LABOUR_MAP = clean_df.set_index("pn").to_dict("index")

LABOUR_MAP


{'FLYC-300-EC-JON16-NS': {'unitsinabox': 25,
  'buildpoints': 6,
  'testonlypoints': 2,
  'gpu': 0,
  'sop': nan,
  'extra': 0.0},
 'N10208GC-10G-ORI02': {'unitsinabox': 1,
  'buildpoints': 15,
  'testonlypoints': 7,
  'gpu': 3,
  'sop': 'Nuvo-10208GC-Outrider-Installation.pptx',
  'extra': 0.0},
 'N8108GC-QD-NM0X': {'unitsinabox': 1,
  'buildpoints': 16,
  'testonlypoints': 5,
  'gpu': 3,
  'sop': 'NoahMedical-8108GC-QD-illustration-SOP.pptx',
  'extra': 5.0},
 'N9160GC-POE-I7DC-4060-CYN02-25': {'unitsinabox': 3,
  'buildpoints': 9,
  'testonlypoints': 5,
  'gpu': 3,
  'sop': 'SF011-A-N9160GC-PoE-i7DC-4060-CYN02-25.V1.1.041224.pdf',
  'extra': 0.0},
 'NRU-110V-AGX32G-COOLIT': {'unitsinabox': 4,
  'buildpoints': 10,
  'testonlypoints': 4,
  'gpu': 0,
  'sop': 'SS001-A-NRU-110V-AGX32G-V1.1.240206.pdf',
  'extra': 0.0},
 'NRU-222S-JAO32G-UBER': {'unitsinabox': 4,
  'buildpoints': 10,
  'testonlypoints': 4,
  'gpu': 0,
  'sop': 'SF020-A-NRU-222S-Uber-V1.3.241202.pdf',
  'extra': 3.0},
 'N

In [7]:
import math
from typing import Any, Dict

def _num(x: Any, default: float = 0.0) -> float:
    """Coerce to float; treat None/NaN/''/'...' as default."""
    try:
        if x is None:
            return default
        # NaN check without pandas
        if isinstance(x, float) and x != x:
            return default
        s = str(x).strip()
        if s in ("", "..."):
            return default
        return float(s)
    except Exception:
        return default

def _normalize_pn(pn: str) -> str:
    # Replace en/em dashes with ASCII dash; upper/strip
    return str(pn).replace("–", "-").replace("—", "-").strip().upper()

def get_cfg_fuzzy(pn_key: str, labour_map: dict) -> dict | None:
    # 1. exact match
    if pn_key in labour_map:
        return labour_map[pn_key]
    # 2. startswith
    for key in labour_map:
        if key.startswith(pn_key):
            return labour_map[key]
    # 3. substring
    for key in labour_map:
        if pn_key in key:
            return labour_map[key]
    return None


def calc_working_hours_single(
    part_number: str,
    units_in_workorder: float,
    assemble: bool,
    gpu_flag: bool,    # True/False
    labour_map: Dict[str, Dict[str, Any]],
):
    """
    Excel-equivalent:
      ((B*IF(C,K,L) + IF(D>0,B*N,0) + IF(sop_flag, B*O, 0) + P
        + IF(B<J,2,ROUNDUP(B/J,0)*2)) / 100) * 8
    Where:
      B=units_in_workorder, C=assemble, D=gpu_flag,
      K=buildpoints, L=testonlypoints, J=unitsinabox, N=gpu, O=sop, P=extra
    """
    try:
        pn_key = _normalize_pn(part_number)
        cfg = get_cfg_fuzzy(pn_key, labour_map)
        if not cfg:
            return "Oha"


        B = _num(units_in_workorder, 0.0)
        if B < 0:
            B = 0.0

        K = _num(cfg.get("buildpoints"), 0.0)        # Assemble=True
        L = _num(cfg.get("testonlypoints"), 0.0)     # Assemble=False
        J = _num(cfg.get("unitsinabox"), 0.0)
        N = _num(cfg.get("gpu"), 0.0)                # GPU rate
        O = _num(cfg.get("sop"), 0.0)                # SOP per-unit rate
        P = _num(cfg.get("extra"), 0.0)              # fixed extra

        term1 = B * (K if assemble else L)
        term2 = (B * N) if gpu_flag else 0.0
        term3 = (B * O) 
        term4 = P
        # IF(B<J, 2, CEIL(B/J)*2) with robust J
        term5 = 2.0 if J <= 0 or B < J else math.ceil(B / J) * 2.0

        return ((term1 + term2 + term3 + term4 + term5) / 100.0) * 8.0
    except Exception:
        return "Oha"

In [ ]:
# Output
hours = calc_working_hours_single(
    part_number="Nuvo-9006DE",
    units_in_workorder=1,
    assemble=True,    
    gpu_flag=False,      
    labour_map=labour_map)

hours

0.88